# TF-IDF + Linear SVM Baseline

Run this notebook from the project root or from the `baselines/` folder. It uses the shared benchmark code, writes artifacts under `artefacts/`, and evaluates `tfidf_linearsvm` on the fixed split.

In [1]:
from pathlib import Path
import sys

PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / "baselines").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent
sys.path.insert(0, str(PROJECT_ROOT))
PROJECT_ROOT

PosixPath('/home/ilya/ML/NLP/project')

In [2]:
MODEL_NAME = "tfidf_linearsvm"
MAX_SAMPLES = None  # set to a small integer, e.g. 3000, for debugging
TOP_GENRES = 15
EPOCHS = 3
BATCH_SIZE = 16
TFIDF_MAX_FEATURES = 100_000

In [3]:
import pandas as pd

from baselines.config import BaselineConfig
from baselines.data_utils import prepare_data
from baselines.run_all import finalize_model_result
from baselines.sklearn_baselines import run_tfidf_linearsvm

config = BaselineConfig(
    max_samples=MAX_SAMPLES,
    top_genres=TOP_GENRES,
    epochs=EPOCHS,
    batch_size=BATCH_SIZE,
    tfidf_max_features=TFIDF_MAX_FEATURES,
)

In [4]:
bundle = prepare_data(config)
bundle.stats

Found candidate tabular files:
  /home/ilya/ML/NLP/project/data/tmdb_movies_2021_2025.csv (80.5 MB)
  /home/ilya/ML/NLP/project/data/tmdb_movies_2021_2025.parquet (50.9 MB)
Choosing largest file by default: /home/ilya/ML/NLP/project/data/tmdb_movies_2021_2025.csv
Prepared data: train=87575, val=34470, test=32986, labels=15, split=temporal_train_le_2023_val_2024_test_2025


{'source_path': '/home/ilya/ML/NLP/project/data/tmdb_movies_2021_2025.csv',
 'detected_columns': {'title': 'title',
  'overview': 'overview',
  'genres': 'genres',
  'release_date': 'release_date',
  'id': 'tmdb_id'},
 'rows_before_filtering': 232586,
 'rows_after_overview_filter': 193927,
 'rows_after_genre_parse': 155678,
 'rows_after_top_genre_filter': 155031,
 'selected_genre_labels': ['Drama',
  'Documentary',
  'Comedy',
  'Horror',
  'Thriller',
  'Animation',
  'Romance',
  'Music',
  'Action',
  'Crime',
  'Fantasy',
  'Science Fiction',
  'Mystery',
  'Family',
  'TV Movie'],
 'per_label_frequency': {'Drama': 54784,
  'Documentary': 43107,
  'Comedy': 29457,
  'Horror': 18098,
  'Thriller': 15386,
  'Animation': 12723,
  'Romance': 10576,
  'Music': 8382,
  'Action': 7238,
  'Crime': 6677,
  'Fantasy': 6339,
  'Science Fiction': 6141,
  'Mystery': 6245,
  'Family': 5088,
  'TV Movie': 3842},
 'train_per_label_frequency': {'Drama': 30116,
  'Documentary': 25421,
  'Comedy': 15

In [5]:
assert bundle.y_train.shape[1] == len(bundle.label_names)
assert bundle.y_val.shape[1] == len(bundle.label_names)
assert bundle.y_test.shape[1] == len(bundle.label_names)
assert {"sample_id", "text", "labels_list"}.issubset(bundle.train_df.columns)
assert bundle.train_df["text"].str.len().min() >= config.min_overview_chars

print("labels:", bundle.label_names)
print("train/val/test:", bundle.y_train.shape, bundle.y_val.shape, bundle.y_test.shape)

labels: ['Drama', 'Documentary', 'Comedy', 'Horror', 'Thriller', 'Animation', 'Romance', 'Music', 'Action', 'Crime', 'Fantasy', 'Science Fiction', 'Mystery', 'Family', 'TV Movie']
train/val/test: (87575, 15) (34470, 15) (32986, 15)


In [6]:
result = run_tfidf_linearsvm(bundle, config)
metrics = finalize_model_result(result, bundle, config)
metrics

{'model': 'tfidf_linearsvm',
 'micro_f1': 0.551087477662032,
 'macro_f1': 0.4418219085292666,
 'weighted_f1': 0.5542022896927753,
 'samples_f1': 0.5666255766707473,
 'precision_micro': 0.4995653754366001,
 'recall_micro': 0.614459002371603,
 'hamming_loss': 0.10407849794862467,
 'precision_at_1': 0.6511853513611835,
 'precision_at_3': 0.3656702843630631,
 'recall_at_3': 0.7605233196574978,
 'subset_accuracy': 0.2504395804280604,
 'average_precision_micro': 0.5719142289292485}

In [7]:
prediction_path = PROJECT_ROOT / "artefacts" / "predictions" / f"{MODEL_NAME}_test_predictions.csv"
threshold_path = PROJECT_ROOT / "artefacts" / "thresholds" / f"{MODEL_NAME}_thresholds.json"
print(prediction_path)
print(threshold_path)
pd.read_csv(prediction_path).head()

/home/ilya/ML/NLP/project/artefacts/predictions/tfidf_linearsvm_test_predictions.csv
/home/ilya/ML/NLP/project/artefacts/thresholds/tfidf_linearsvm_thresholds.json


,sample_id,text,true_labels,predicted_labels,score_drama,score_documentary,score_comedy,score_horror,score_thriller,score_animation,score_romance,score_music,score_action,score_crime,score_fantasy,score_science_fiction,score_mystery,score_family,score_tv_movie
0,1052558,iPossessed [SEP] A group of celebrating friend...,Horror|Thriller,Horror|Thriller|Fantasy,-1.021572,-1.293990,-0.928889,1.768997,0.339596,-1.428571,-1.250349,-0.727251,-0.708538,-0.899003,0.246519,-1.140384,-1.067935,-1.420144,-1.208031
1,980477,Ne Zha 2 [SEP] After a catastrophic event leav...,Animation|Action|Fantasy,Drama|Action|Science Fiction,0.078168,-1.693813,-0.500230,-1.994593,-1.204551,-1.051573,-0.223007,-0.669113,0.362142,-1.188691,-0.342417,0.358950,-1.926172,-1.023130,-1.757975
2,1205229,Night of the Zoopocalypse [SEP] A wolf and mou...,Comedy|Horror|Animation|Science Fiction,Horror|Animation|Action,-1.994558,-1.298922,-0.700469,0.420833,-0.578460,0.796331,-1.814081,-1.471821,0.450457,-1.465790,-1.126721,-0.046412,-2.091796,-0.306778,-0.447518
3,1084199,Companion [SEP] During a weekend getaway at a ...,Horror|Thriller|Science Fiction,Drama|Thriller|Science Fiction,0.142967,-1.525913,-0.530146,-0.128074,0.947681,-2.145035,-0.385948,-0.972448,-1.061197,-0.587721,-1.754115,0.005787,-0.413210,-2.386077,-1.519439
4,1009640,Valiant One [SEP] With tensions between North ...,Thriller|Action,Action,-0.284352,-0.924218,-0.739585,-0.674804,-0.470156,-0.957116,-0.491561,-1.654228,0.237638,-0.805292,-1.630432,-0.562812,-1.136713,-1.788967,-1.125564
